# Database preparation

Create sql connection

In [95]:
import sqlite3

# connect to database (this creates gym.db if it does not exist)
conn = sqlite3.connect("gym.db")

# cursor to execute SQL
cur = conn.cursor()

Execute schema and data to make the database ready for use

In [96]:
# Execute schema.sql
with open("schema.sql", "r", encoding="utf-8") as f:  # Reads the file
    schema_sql = f.read()

cur.executescript(schema_sql)  # runs sql statements
conn.commit()  # saves changes

In [97]:
# Execute data.sql
with open("data.sql", "r", encoding="utf-8") as f:
    data_sql = f.read()

cur.executescript(data_sql)
conn.commit()

Now we can query from the database

In [98]:
# Query the database
cur.execute("SELECT * FROM users")

rows = cur.fetchall()
for row in rows:
    print(row)

(1, 'Johnny', 'Nordmann', 'johnny@stud.ntnu.no', '40100001', 1)
(2, 'Viola', 'Smith', 'viola@stud.ntnu.no', '40100010', 1)
(3, 'Erik', 'Hansen', 'erik.hansen@stud.ntnu.no', '40100002', 1)
(4, 'Sara', 'Berg', 'sara.berg@stud.ntnu.no', '40100003', 1)
(5, 'Lucas', 'Johansen', 'lucas.johansen@stud.ntnu.no', '40100004', 0)
(6, 'Emma', 'Larsen', 'emma.larsen@stud.ntnu.no', '40100005', 1)
(7, 'Noah', 'Dahl', 'noah.dahl@stud.ntnu.no', '40100006', 0)
(8, 'Maja', 'Nilsen', 'maja.nilsen@stud.ntnu.no', '40100007', 1)
(9, 'Oliver', 'Strand', 'oliver.strand@stud.ntnu.no', '40100008', 0)
(10, 'Ingrid', 'Moen', 'ingrid.moen@stud.ntnu.no', '40100009', 1)


In [110]:
# Query the database
cur.execute("SELECT * FROM session_bookings")

rows = cur.fetchall()
for row in rows:
    print(row)

(1, 3, 2, '2026-03-15 18:52:55')


# Interface

We can use Python as application layer to receive user input, call SQL queries, and show results, i.e. an interface between user and database.

## Book a session

### Show available

In [100]:
import pandas as pd
query = """SELECT 
                g.gym_name AS Gym,
                a.name,
                a.description,
                s.sessiondate AS Day,
                s.start_time AS Time
                --s.signup_deadline
            FROM group_sessions s
            INNER JOIN activities a ON a.id = s.activity_id
            INNER JOIN halls h ON h.id = s.hall_id
            INNER JOIN gyms g ON g.id = h.gym_id"""

df = pd.read_sql_query(query, conn)

df

,Gym,name,description,Day,Time
0,Øya,Spin 8x3min,Intervalltime på sykkel der du jobber i interv...,2026-03-16,18:00
1,Dragvoll,Spin45,Spinningtime med variert løype fordelt på 2-3 ...,2026-03-16,16:00
2,Øya,Spin60,Spinningtime med variert løype fordelt på 2-4 ...,2026-03-17,18:30
3,Gloeshaugen,Spin 4x4,"Intervalltime på sykkel med god oppvarming, de...",2026-03-17,17:00
4,Øya,Spin 4x4,"Intervalltime på sykkel med god oppvarming, de...",2026-03-18,19:00
5,Dragvoll,Spin45,Spinningtime med variert løype fordelt på 2-3 ...,2026-03-18,16:00


### Book
Plan: User must first enter email to "log in", email corresponds to a unique user_id as well (users might change their emails, so then they do not lose their data). After identification, user chooses class from the timetable. System checks that user is not banned (do not have three dots), then booking is performed using user_id and session_id. 

In [101]:
from datetime import datetime
booking_time = datetime.now()
datetime_object = str((datetime.now()).replace(microsecond=0))
date_part = booking_time.date()
time_part = (booking_time.time()).replace(microsecond=0)
# time_part = (booking_time.time()).replace(second=0, microsecond=0)
print(booking_time)
print(date_part)
print(time_part)
print(datetime_object)
print(type(datetime_object))


2026-03-15 18:52:05.422460
2026-03-15
18:52:05
2026-03-15 18:52:05
<class 'str'>


Plan: User inserts their username (email), which is unique in the system. This email corresponds to a unique ID that is returned. This ID is further used in the booking. I do this because it simplifies stuff since user_id is used elsewhere

In [103]:
# Write in username (email), this is unique in the system so it corresponds to a unique user_id

email = str(input("Enter username: "))

cur.execute("""SELECT id FROM users WHERE email = ? ;""", (email,))
email_to_id = cur.fetchone()[0]

print(email_to_id)

2


User has to choose which class to go to, this give session_id. Since this will only be in CLI, I show numbered options and let user choose a number, since typing correct input can be annoying for user.

In [104]:
cur.execute("""SELECT distinct sessiondate FROM group_sessions ORDER BY sessiondate""")

dates = cur.fetchall()

print("Please select date by entering the number on the left:")
for i, d in enumerate(dates):
    print(i + 1, d[0])  # count from 1

date = int(input("Choose date: "))
selected_date = dates[date - 1][0]  # To take only the date string from a list of tuples on the form [(), ()], counting from zero

print("Selected date: ", selected_date)

Please select date by entering the number on the left:
1 2026-03-16
2 2026-03-17
3 2026-03-18
Selected date:  2026-03-17


In [105]:
df = pd.read_sql_query("""SELECT name, start_time, description, signup_deadline FROM activities INNER JOIN group_sessions ON activities.id = group_sessions.activity_id WHERE sessiondate='2026-03-17' ORDER BY group_sessions.start_time""", conn)

df

,name,start_time,description,signup_deadline
0,Spin 4x4,17:00,"Intervalltime på sykkel med god oppvarming, de...",16:00
1,Spin60,18:30,Spinningtime med variert løype fordelt på 2-4 ...,17:30


In [106]:
cur.execute("""SELECT g.id, a.name, g.start_time, a.description, g.signup_deadline FROM activities a INNER JOIN group_sessions g ON a.id = g.activity_id WHERE sessiondate=? ORDER BY g.start_time""", (selected_date,))

trainings = cur.fetchall()

print("Please training by entering the number on the left:")
for i, (session_id, name, start_time, description, signup_deadline) in enumerate(trainings, start=1):
    print(f"{i}. {name}, starting {start_time}, signup deadline {signup_deadline}:")
    print(f"    {description}")
    print("     ")

training = int(input("Choose date: "))
selected_training = trainings[training - 1]  # To take only the training string from a list of tuples on the form [(), ()], counting from zero

print("Selected training: ", selected_training)

session_id, name, start_time, description, signup_deadline = selected_training

print(name)
print(session_id)

Please training by entering the number on the left:
1. Spin 4x4, starting 17:00, signup deadline 16:00:
    Intervalltime på sykkel med god oppvarming, deretter 4 stående intervaller på 4 minutter med ca 2 minutter aktiv pause mellom hvert drag. Timen avsluttes med nedsykling.
     
2. Spin60, starting 18:30, signup deadline 17:30:
    Spinningtime med variert løype fordelt på 2-4 arbeidsperioder. Litt raskere tempo og lengre stående perioder enn Spin45, men du bestemmer selv intensiteten med motstanden du legger på.
     
Selected training:  (3, 'Spin60', '18:30', 'Spinningtime med variert løype fordelt på 2-4 arbeidsperioder. Litt raskere tempo og lengre stående perioder enn Spin45, men du bestemmer selv intensiteten med motstanden du legger på.', '17:30')
Spin60
3


In [107]:
#### Book session
from datetime import datetime


def book_session(user_id, session_id):
    '''
    Function that performs a booking
    --------------------
    user_id : Users unique id
    session_id: Sessions unique id (is retrieved from ****)
    '''

    #### First check if available spots by comparing booked spots with capacity
    cur.execute("""SELECT COUNT(*) FROM session_bookings WHERE session_id = ?;""", (session_id,))
    booked = cur.fetchone()[0]

    cur.execute("""SELECT capacity FROM halls JOIN group_sessions ON halls.id = group_sessions.hall_id WHERE group_sessions.id = ?;""", (session_id,))

    capacity = cur.fetchone()[0]

    if booked >= capacity:
        print("Session is full")
        return

    # If did not return above, means it was available spots so booking is performed:
    booking_time = str((datetime.now()).replace(microsecond=0))  # Booking time in format YYY-MM-DD HH:MM:SS

    cur.execute("""INSERT INTO session_bookings (session_id, user_id, booking_time) VALUES (?, ?, ?);""", (session_id, user_id, booking_time))

    conn.commit()

    print("Booking successful")

### Booking example: Johnny

In [108]:
print(type(email_to_id))
print(type(session_id))

<class 'int'>
<class 'int'>


In [109]:
book_session(email_to_id, session_id)

Booking successful


#### Email to ID

In [17]:
# Write in username (email), this is unique in the system so it corresponds to a unique user_id

email = 'johnny@stud.ntnu.no'

cur.execute("""SELECT id FROM users WHERE email = ? ;""", (email,))
email_to_id = cur.fetchone()[0]

print(email_to_id)

1


In [ ]:
#### Register when user doesn't show up

In [ ]:
#### Check if user has three dots and should be blacklisted

In [ ]:
#### 

# Reset database during development for recreation

In [94]:
#### Reset database during development so that we can recreate it from scratch when needed

import os

conn.close()  # Must remove db connection first

if os.path.exists("gym.db"):
    os.remove("gym.db")